# Repeater-direction check, Cat 2, step 1 (Mahalanobis membership)

The four candidate non-repeaters scored at the very top of the blind anomaly hunt. That on its own only says they are extreme; it does not say they are extreme *in the repeater way*. This notebook asks the question: does each candidate look like it belongs to the repeater population?

**Step 1 (this notebook):** model the repeater population as one cloud and measure each burst's Mahalanobis distance to it. Then judge that distance against a non-arbitrary benchmark, namely how far real repeaters sit from their own cloud, so the cut-off is a percentile of a real population rather than a number picked by hand.

Run the cells top to bottom. Each code cell has a plain-English cell above it saying what it does and how to read the output.

## 1. Load the standardised features

Read `catalog2_features_scaled.csv`, the cleaned and standardised feature table the cleaning notebook produced upstream. Standardised so the raw numbers are directly comparable across features.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Find the project root by walking up until see data/processed/phase_1.
# Keeps the notebook runnable wherever Jupyter was started, instead of a fragile
# relative path.
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / 'data' / 'processed' / 'phase_1').exists():
            return p
    raise FileNotFoundError('Could not find data/processed/phase_1 above ' + str(start))

ROOT = find_project_root()
SCALED = ROOT / 'data' / 'processed' / 'phase_1' / 'catalog2_features_scaled.csv'

df = pd.read_csv(SCALED)
print('loaded:', SCALED.name)
print('rows, cols:', df.shape)
print('columns:', list(df.columns))

loaded: catalog2_features_scaled.csv
rows, cols: (4109, 13)
columns: ['tns_name', 'sub_num', 'is_repeater', 'repeater_name', 'catalog1_flag', 'dm_fitb', 'width_fitb', 'flux', 'fluence', 'sp_idx', 'sp_run', 'peak_freq', 'bandwidth']


## 2. Split repeaters vs non-repeaters and build the feature matrix

`X` is every burst as a row of eight features. Split it into the repeaters (our known-odd reference population) and the non-repeaters (which the candidates belong to). The four candidates are named here; score every non-repeater but spotlight these at the end.

In [2]:
# The eight standardised morphological features (already mean 0, unit spread).
# sp_idx/sp_run reinstated 2026-06-26 as narrowband proxies (see notebook 01 / Catalog 2).
FEATURES = ['dm_fitb', 'width_fitb', 'flux', 'fluence',
            'sp_idx', 'sp_run', 'peak_freq', 'bandwidth']

# Repeater flag. pandas usually reads this column straight to bool, but coerce
# defensively: astype(bool) on the text False would wrongly come out True, so map
# the values explicitly instead of trusting the dtype.
is_rep = df['is_repeater'].map({True: True, False: False,
                                'True': True, 'False': False}).fillna(False).astype(bool)

X = df[FEATURES].to_numpy(dtype=float)       # every burst, eight features
X_rep = X[is_rep.to_numpy()]                 # repeaters only
X_non = X[(~is_rep).to_numpy()]              # non-repeaters only

print('all bursts    :', X.shape[0])
print('repeaters     :', X_rep.shape[0])
print('non-repeaters :', X_non.shape[0])

# The four candidate non-repeaters from the Phase 1 plan (top of the blind ranking,
# all three methods agreeing). Score every non-repeater but spotlight these.
CANDIDATES = ['FRB20200321E', 'FRB20181119D', 'FRB20200124E', 'FRB20201119A']

present = set(df['tns_name'])
for name in CANDIDATES:
    if name not in present:
        print('WARNING: candidate not found in cleaned table:', name)

all bursts    : 4109
repeaters     : 1047
non-repeaters : 3062


## 3. Model the repeater cloud (Mahalanobis)

Fit a single cloud to the repeaters: its centre (mean) and its shape and spread (covariance). Mahalanobis distance then measures how far any burst sits from that cloud, in units that account for the cloud being wider in some directions than others. A small distance means the burst looks like it came from the repeater population.

Note on the choice: this uses the plain empirical covariance for full transparency. It assumes the repeaters form one roughly elliptical form. If they are actually several sub-groups, that assumption does not hold, which is exactly why step 2 adds LOF and EIF, two methods that do not assume a single form.

In [3]:
from sklearn.covariance import EmpiricalCovariance

# Model the repeater population as one cloud: its mean (centre) and covariance
# (shape and spread). EmpiricalCovariance is the plain, fully transparent fit, it
# uses every repeater equally and makes no robust down-weighting choices.
# MinCovDet would be the robust alternative (fit only the tightest core of
# repeaters); noted as a later option.
rep_cloud = EmpiricalCovariance().fit(X_rep)

# .mahalanobis returns the SQUARED Mahalanobis distance; take the root so the
# number reads as a distance in standard-deviation-like units.
d_all = np.sqrt(rep_cloud.mahalanobis(X))      # every burst to the repeater cloud
d_rep = np.sqrt(rep_cloud.mahalanobis(X_rep))  # repeaters to their own cloud
d_non = np.sqrt(rep_cloud.mahalanobis(X_non))  # non-repeaters to the repeater cloud

print('median distance to the repeater cloud:')
print('  repeaters     :', round(float(np.median(d_rep)), 2))
print('  non-repeaters :', round(float(np.median(d_non)), 2))

median distance to the repeater cloud:
  repeaters     : 2.16
  non-repeaters : 4.24


## 4. The benchmark: how far do real repeaters sit from their own cloud?

Rather than inventing a distance cut-off, use the repeaters themselves. For any distance ask: what fraction of real repeaters sit at least as close to the cloud as this? A burst at the median repeater distance scores 0.50; one out at the edge of the repeater spread scores around 0.95; one farther out than essentially every repeater scores near 1.0 and is not repeater-like. The natural, non-arbitrary line is the 95th percentile of the repeaters' own distances: inside it, the burst is as close to the cloud as a real repeater would be.

In [4]:
def repeater_percentile(d):
    # Fraction of REAL repeaters sitting at least as close to the cloud as distance d.
    # 0.50 = as central as a typical repeater; ~0.95 = out at the edge of the
    # repeater spread; near 1.0 = farther out than essentially every repeater (so
    # not repeater-like). Small values = deep in the repeater core.
    return float(np.mean(d_rep <= d))

# Sanity: how many non-repeaters even fall inside the repeater spread (pctile <= 0.95)?
frac = float(np.mean([repeater_percentile(d) <= 0.95 for d in d_non]))
print('non-repeaters inside the repeater spread (percentile <= 0.95):',
      round(frac * 100, 1), 'percent')

non-repeaters inside the repeater spread (percentile <= 0.95): 60.2 percent


## 5. Score every non-repeater and spotlight the candidates

Compute the distance and the repeater-percentile for every non-repeater, rank them closest-first, then pull out the four candidates. Lower distance and lower percentile mean more repeater-like. The rank (out of all non-repeaters) tells you how unusual it is for a non-repeater to look this much like a repeater.

In [5]:
# Score every non-repeater, then rank by how repeater-like it is (closest first).
non = df.loc[(~is_rep).to_numpy(), ['tns_name', 'sub_num']].copy()
non['distance'] = d_non
non['rep_percentile'] = [repeater_percentile(d) for d in d_non]
non = non.sort_values('distance').reset_index(drop=True)
non['rank'] = np.arange(1, len(non) + 1)
n_non = len(non)
print('scored', n_non, 'non-repeaters')
print()

print('CANDIDATE SPOTLIGHT (lower distance / lower percentile = more repeater-like)')
spot = non[non['tns_name'].isin(CANDIDATES)].sort_values('rank')
for _, r in spot.iterrows():
    nm, sb = r['tns_name'], int(r['sub_num'])
    ds, pc, rk = r['distance'], r['rep_percentile'], int(r['rank'])
    print(f'  {nm:14s} sub {sb}: distance {ds:5.2f}, repeater-percentile {pc:.2f}, rank {rk} of {n_non}')

scored 3062 non-repeaters

CANDIDATE SPOTLIGHT (lower distance / lower percentile = more repeater-like)
  FRB20200321E   sub 1: distance  2.46, repeater-percentile 0.61, rank 317 of 3062
  FRB20181119D   sub 2: distance  2.71, repeater-percentile 0.70, rank 454 of 3062
  FRB20181119D   sub 0: distance  2.84, repeater-percentile 0.72, rank 530 of 3062
  FRB20200124E   sub 0: distance  4.32, repeater-percentile 0.93, rank 1612 of 3062
  FRB20201119A   sub 0: distance  4.42, repeater-percentile 0.94, rank 1705 of 3062
  FRB20181119D   sub 1: distance  4.46, repeater-percentile 0.95, rank 1743 of 3062
  FRB20200321E   sub 0: distance  4.51, repeater-percentile 0.95, rank 1804 of 3062


## How to read the result, and what comes next

- **Low percentile (say under 0.95) and a low rank:** the candidate sits inside the repeater spread, i.e. it is odd in the repeater direction. That would be the interesting case, the repeater-not-yet-seen-to-repeat hypothesis.
- **High percentile (near 1.0):** the candidate is extreme, but not (on distance alone) in the repeater way. Section 6 checks whether that is a genuinely different direction or just farther along the repeater direction.

**Caveats baked in here on purpose, not hidden:**
1. The cloud is fit on repeater *sub-bursts*, so prolific repeaters with many sub-bursts pull the cloud toward themselves (the source-correlation issue your validation handled by grouping per source). If a candidate looks borderline, refit on per-source averaged repeaters and compare.
2. Empirical covariance assumes one elliptical blob. The LOF/EIF robustness check against the same repeater set drops that assumption; agreement across all three is what makes a verdict trustworthy, exactly as in the original hunt.
3. This reads membership, not direction *per feature*. Section 6 adds the which-features-make-it-repeater-like read.

Sections 6 and 7 below take this further: the feature-direction read, then a contamination check on the whole list. The LOF/EIF robustness check comes after those.

## 6. Step 3: which features drive the candidates, and do they point the repeater way?

Step 1 indicated the candidates sit *far* from the repeater cloud, but distance alone cannot say whether that is the same kind of weird as repeaters or a different kind. This cell answers that.

First it builds the **repeater signature**: the average standardised value of each feature across repeaters, i.e. which features repeaters run high or low on. Then for each candidate sub-burst it prints the full feature fingerprint, marks the single feature driving it hardest, and computes the **cosine** between the candidate and the repeater signature: +1 means pointing exactly the repeater way, 0 unrelated, -1 opposite. That cosine is judged against the range real repeaters themselves show.

Two things to look for: (1) does the candidate point the repeater way or a different way; (2) is one feature sitting at a wild value (beyond about +/-6) while the rest are ordinary, which would mean the extremeness is one suspicious number, possibly a placeholder, worth checking against the raw unscaled value.

In [6]:
# The repeater signature: average standardised value per feature. The whole-catalogue
# average is ~0 because the features are globally standardised, so this IS how
# repeaters sit relative to everything else.
rep_dir = X_rep.mean(axis=0)

def cosine(a, b):
    # +1 same direction, 0 unrelated, -1 opposite. Direction only, ignores distance.
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return float('nan')
    return float(np.dot(a, b) / (na * nb))

# Reference: how aligned a REAL repeater is with the repeater signature, so judge
# the candidates against the repeaters themselves, not a hand-picked number.
rep_cos = np.array([cosine(x, rep_dir) for x in X_rep])
rep_lo = float(np.nanpercentile(rep_cos, 5))
print('repeater alignment with their own signature (cosine):',
      'median', round(float(np.nanmedian(rep_cos)), 2),
      ', 5th percentile', round(rep_lo, 2))
print()

print('REPEATER SIGNATURE (average standardised value; + runs high, - runs low):')
for feat, v in zip(FEATURES, rep_dir):
    print(f'  {feat:10s} {v:+.2f}')
print()

print('CANDIDATES (cosine vs the repeater signature, then the feature fingerprint):')
for name in CANDIDATES:
    for i in df.index[df['tns_name'] == name]:
        x = X[i]
        c = cosine(x, rep_dir)
        biggest = int(np.argmax(np.abs(x)))
        sb = int(df.loc[i, 'sub_num'])
        tag = 'points the repeater way' if c >= rep_lo else 'points a different way'
        print()
        print(f'{name} sub {sb}:  cosine {c:+.2f}  [{tag}]')
        for j, (feat, v) in enumerate(zip(FEATURES, x)):
            mark = '  <-- biggest' if j == biggest else ''
            print(f'    {feat:10s} {v:+6.2f}{mark}')

repeater alignment with their own signature (cosine): median 0.76 , 5th percentile 0.03

REPEATER SIGNATURE (average standardised value; + runs high, - runs low):
  dm_fitb    -0.75
  width_fitb +0.75
  flux       -0.29
  fluence    +0.12
  sp_idx     +0.81
  sp_run     -0.84
  peak_freq  +0.21
  bandwidth  -0.78

CANDIDATES (cosine vs the repeater signature, then the feature fingerprint):

FRB20200321E sub 0:  cosine +0.76  [points the repeater way]
    dm_fitb     +0.80
    width_fitb  +0.87
    flux        -0.55
    fluence     -0.53
    sp_idx      +3.04  <-- biggest
    sp_run      -2.94
    peak_freq   +0.99
    bandwidth   -1.81

FRB20200321E sub 1:  cosine +0.57  [points the repeater way]
    dm_fitb     +0.80
    width_fitb  +0.56
    flux        -0.55
    fluence     -0.53
    sp_idx      +1.37
    sp_run      -0.98
    peak_freq   +1.41  <-- biggest
    bandwidth   -0.70

FRB20181119D sub 0:  cosine +0.74  [points the repeater way]
    dm_fitb     +0.23
    width_fitb  +1.76

## 7. How contaminated is the whole anomaly list?

The four candidates turned out to be spectral-fit blowups. This cell asks whether that is the whole top of the ranking, not just those four. For every burst it finds the single feature with the largest standardised value (the one driving its anomaly), flags it if that feature is `sp_idx` or `sp_run`, then reports what fraction of the top anomalies (ranked by the saved `maha_score` from the original hunt) are spectral-fit-driven. A high fraction means sp_idx and sp_run are poisoning the hunt and need the same treatment scattering got.

In [7]:
# Bring in the saved anomaly scores from the original hunt so one can rank by them.
scores = pd.read_csv(ROOT / 'data' / 'processed' / 'phase_1' / 'catalog2_method_scores.csv')
m = df.merge(scores[['tns_name', 'sub_num', 'maha_score', 'lof_median_rank', 'eif_score']],
             on=['tns_name', 'sub_num'], how='left')

# For every burst, find the single feature with the largest standardised magnitude:
# that is the feature driving its anomaly. Flag it if that feature is spectral.
Z = m[FEATURES].to_numpy(dtype=float)
dom_idx = np.argmax(np.abs(Z), axis=1)
dom_feat = np.array(FEATURES)[dom_idx]
spectral = np.isin(dom_feat, ['sp_idx', 'sp_run'])

print('OVERALL')
print('  bursts whose single biggest feature is sp_idx or sp_run:',
      int(spectral.sum()), 'of', len(m),
      '(' + str(round(100 * spectral.mean(), 1)) + ' percent)')
extreme_spec = (np.abs(m['sp_idx'].to_numpy()) > 5) | (np.abs(m['sp_run'].to_numpy()) > 5)
print('  bursts with |sp_idx| or |sp_run| > 5 (clear fit blowups):',
      int(extreme_spec.sum()), 'of', len(m))
print()

# Rank by the saved maha_score (higher = more anomalous). NaN scores pushed to the bottom.
ms = m['maha_score'].to_numpy(dtype=float)
order = np.argsort(np.where(np.isnan(ms), -np.inf, ms))[::-1]

print('CONTAMINATION OF THE TOP ANOMALIES (ranked by maha_score):')
for K in [20, 50, 100]:
    frac = spectral[order[:K]].mean()
    print('  top', K, 'anomalies:', round(100 * frac, 1), 'percent driven by sp_idx / sp_run')
print()

print('TOP 15 ANOMALIES, with the feature driving each:')
for r in order[:15]:
    row = m.iloc[r]
    nm, sb = row['tns_name'], int(row['sub_num'])
    feat, z = dom_feat[r], Z[r, dom_idx[r]]
    print(f'  {nm:14s} sub {sb}: driven by {feat:10s} ({z:+.1f})')

OVERALL
  bursts whose single biggest feature is sp_idx or sp_run: 707 of 4109 (17.2 percent)
  bursts with |sp_idx| or |sp_run| > 5 (clear fit blowups): 0 of 4109

CONTAMINATION OF THE TOP ANOMALIES (ranked by maha_score):
  top 20 anomalies: 80.0 percent driven by sp_idx / sp_run
  top 50 anomalies: 72.0 percent driven by sp_idx / sp_run
  top 100 anomalies: 70.0 percent driven by sp_idx / sp_run

TOP 15 ANOMALIES, with the feature driving each:
  FRB20221219D   sub 0: driven by sp_run     (+3.9)
  FRB20200705D   sub 0: driven by sp_run     (+3.6)
  FRB20230422A   sub 0: driven by sp_run     (+3.6)
  FRB20190531E   sub 0: driven by sp_run     (+3.1)
  FRB20211104E   sub 0: driven by sp_idx     (-1.8)
  FRB20190805E   sub 1: driven by sp_idx     (-2.3)
  FRB20230908C   sub 0: driven by sp_run     (+3.2)
  FRB20210222C   sub 0: driven by sp_run     (+3.4)
  FRB20200429A   sub 0: driven by sp_run     (+3.3)
  FRB20210512C   sub 0: driven by sp_run     (+3.3)
  FRB20190104B   sub 0: driv